In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install tensorflow tensorflow_hub tensorflow_datasets -q

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
!unzip "/content/drive/MyDrive/butterfly.zip" -d "/content/butterfly/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/butterfly/train/Image_2348.jpg  
  inflating: /content/butterfly/train/Image_2349.jpg  
  inflating: /content/butterfly/train/Image_235.jpg  
  inflating: /content/butterfly/train/Image_2350.jpg  
  inflating: /content/butterfly/train/Image_2351.jpg  
  inflating: /content/butterfly/train/Image_2352.jpg  
  inflating: /content/butterfly/train/Image_2353.jpg  
  inflating: /content/butterfly/train/Image_2354.jpg  
  inflating: /content/butterfly/train/Image_2355.jpg  
  inflating: /content/butterfly/train/Image_2356.jpg  
  inflating: /content/butterfly/train/Image_2357.jpg  
  inflating: /content/butterfly/train/Image_2358.jpg  
  inflating: /content/butterfly/train/Image_2359.jpg  
  inflating: /content/butterfly/train/Image_236.jpg  
  inflating: /content/butterfly/train/Image_2360.jpg  
  inflating: /content/butterfly/train/Image_2361.jpg  
  inflating: /content/butterfly/train/Image_2362.jpg  
  inflating: /co

In [ ]:
data_dir = "/content/butterfly/train"  # images are here
train_csv = "/content/drive/MyDrive/dl_train_val_set.csv"
test_csv = "/content/drive/MyDrive/dl_test_set.csv"

In [ ]:
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
print("Train CSV columns:", train_df.columns)
print("Test CSV columns:", test_df.columns)

Train CSV columns: Index(['Unnamed: 0', 'filename', 'label'], dtype='object')
Test CSV columns: Index(['Unnamed: 0', 'filename', 'label'], dtype='object')


In [ ]:
# Keep only the last two columns (filename, label)
train_df = train_df.iloc[:, 1:3]
test_df = test_df.iloc[:, 1:3]

# Rename columns properly
train_df.columns = ['filename', 'label']
test_df.columns = ['filename', 'label']


In [ ]:
train_df['filename'] = train_df['filename'].apply(lambda x: os.path.join(data_dir, x))
test_df['filename'] = test_df['filename'].apply(lambda x: os.path.join(data_dir, x))

In [ ]:
le = LabelEncoder()
train_df['label_enc'] = le.fit_transform(train_df['label'])
test_df['label_enc'] = le.transform(test_df['label'])

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='label',
    subset='training',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='label',
    subset='validation',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 4900 validated image filenames belonging to 75 classes.
Found 1224 validated image filenames belonging to 75 classes.
Found 375 validated image filenames belonging to 75 classes.


In [ ]:
# model
base_model = EfficientNetB0(weights='imagenet',include_top=False,input_shape=(224,224,3)
)
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
preds = Dense(len(le.classes_), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Freeze base layers for transfer learning
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# ---Train Model ---
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)


Epoch 1/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 137s 701ms/step - accuracy: 0.4498 - loss: 2.8298 - val_accuracy: 0.7508 - val_loss: 1.6902
Epoch 2/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 73s 474ms/step - accuracy: 0.7839 - loss: 1.3382 - val_accuracy: 0.8292 - val_loss: 1.0317
Epoch 3/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 72s 470ms/step - accuracy: 0.8471 - loss: 0.8850 - val_accuracy: 0.8399 - val_loss: 0.7871
Epoch 4/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 73s 474ms/step - accuracy: 0.8753 - loss: 0.6750 - val_accuracy: 0.8709 - val_loss: 0.6598
Epoch 5/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 73s 480ms/step - accuracy: 0.8924 - loss: 0.5633 - val_accuracy: 0.8644 - val_loss: 0.5941
Epoch 6/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 73s 473ms/step - accuracy: 0.9090 - loss: 0.4747 - val_accuracy: 0.8807 - val_loss: 0.5305
Epoch 7/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 73s 471ms/step - accuracy: 0.9171 - loss: 0.4226 - val_accuracy: 0.9020 - val_loss: 0.4926
Epoch 8/10
154/154 ━━━━━━━━━━━━━━━━━━━━ 72s 467ms/step - accuracy: 0.9247 - loss: 

In [ ]:
# ---Fine-Tuning the Model ---

for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    verbose=1
)


Epoch 1/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 130s 661ms/step - accuracy: 0.8351 - loss: 0.7385 - val_accuracy: 0.8611 - val_loss: 0.5560
Epoch 2/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 76s 496ms/step - accuracy: 0.8647 - loss: 0.6083 - val_accuracy: 0.8660 - val_loss: 0.5629
Epoch 3/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 74s 479ms/step - accuracy: 0.8808 - loss: 0.5421 - val_accuracy: 0.8758 - val_loss: 0.5417
Epoch 4/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 74s 481ms/step - accuracy: 0.9010 - loss: 0.4639 - val_accuracy: 0.8791 - val_loss: 0.5111
Epoch 5/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 80s 523ms/step - accuracy: 0.9071 - loss: 0.4340 - val_accuracy: 0.8930 - val_loss: 0.4866


In [ ]:
# --- Evaluate on Test Set ---
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\n Test Accuracy: {test_acc * 100:.2f}%")

11/12 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.9098 - loss: 0.3536

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- Predictions ---
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

annot_data = np.asarray([f'{val}' if val > 0 else '' for val in cm.flatten()]).reshape(cm.shape)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=annot_data,       # Use the custom annotation data
    fmt='',                 # Set format to empty since annotations are already strings
    cbar=False,             # This removes the color bar on the side
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Butterfly Species Classification — Confusion Matrix")
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# --- Predictions ---
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

# --- Metrics (in percentage) ---
acc = accuracy_score(y_true, y_pred) * 100
prec = precision_score(y_true, y_pred, average='weighted') * 100
rec = recall_score(y_true, y_pred, average='weighted') * 100
f1 = f1_score(y_true, y_pred, average='weighted') * 100

print(f"Accuracy: {acc:.2f}%")
print(f"Precision: {prec:.2f}%")
print(f"Recall: {rec:.2f}%")
print(f"F1 Score: {f1:.2f}%")


12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step
Accuracy: 90.13%
Precision: 90.88%
Recall: 90.13%
F1 Score: 89.86%


In [ ]:
class_report = classification_report(y_true, y_pred, target_names=le.classes_)
print("Classification Report:")
print(class_report)

Classification Report:
                           precision    recall  f1-score   support

                   ADONIS       1.00      0.60      0.75         5
AFRICAN GIANT SWALLOWTAIL       1.00      1.00      1.00         5
           AMERICAN SNOOT       1.00      1.00      1.00         5
                    AN 88       0.83      1.00      0.91         5
                  APPOLLO       1.00      1.00      1.00         5
                    ATALA       1.00      1.00      1.00         5
 BANDED ORANGE HELICONIAN       0.80      0.80      0.80         5
           BANDED PEACOCK       1.00      0.80      0.89         5
            BECKERS WHITE       1.00      0.80      0.89         5
         BLACK HAIRSTREAK       1.00      1.00      1.00         5
              BLUE MORPHO       1.00      1.00      1.00         5
        BLUE SPOTTED CROW       0.83      1.00      0.91         5
           BROWN SIPROETA       1.00      1.00      1.00         5
            CABBAGE WHITE       0.71  

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Get the full report as a dictionary instead of a string
report_dict = classification_report(y_true, y_pred, target_names=le.classes_, output_dict=True)

# Convert to DataFrame for easier filtering
report_df = pd.DataFrame(report_dict).transpose()

# Keep only first 3 species + the last 3 summary rows
summary = pd.concat([report_df.iloc[:3], report_df.tail(3)])

# Display neatly
print("Simplified Classification Report:\n")
print(summary[['precision', 'recall', 'f1-score', 'support']])

Simplified Classification Report:

                           precision    recall  f1-score     support
ADONIS                      1.000000  0.600000  0.750000    5.000000
AFRICAN GIANT SWALLOWTAIL   1.000000  1.000000  1.000000    5.000000
AMERICAN SNOOT              1.000000  1.000000  1.000000    5.000000
accuracy                    0.901333  0.901333  0.901333    0.901333
macro avg                   0.908825  0.901333  0.898566  375.000000
weighted avg                0.908825  0.901333  0.898566  375.000000


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Get the full report as a dictionary instead of a string
report_dict = classification_report(y_true, y_pred, target_names=le.classes_, output_dict=True)

# Convert to DataFrame for easier filtering
report_df = pd.DataFrame(report_dict).transpose()

# Keep only first 3 species + the last 3 summary rows
summary = pd.concat([report_df.iloc[:25], report_df.tail(3)])

# Display neatly
print("Simplified Classification Report:\n")
print(summary[['precision', 'recall', 'f1-score', 'support']])

Simplified Classification Report:

                           precision    recall  f1-score     support
ADONIS                      1.000000  0.600000  0.750000    5.000000
AFRICAN GIANT SWALLOWTAIL   1.000000  1.000000  1.000000    5.000000
AMERICAN SNOOT              1.000000  1.000000  1.000000    5.000000
AN 88                       0.833333  1.000000  0.909091    5.000000
APPOLLO                     1.000000  1.000000  1.000000    5.000000
ATALA                       1.000000  1.000000  1.000000    5.000000
BANDED ORANGE HELICONIAN    0.800000  0.800000  0.800000    5.000000
BANDED PEACOCK              1.000000  0.800000  0.888889    5.000000
BECKERS WHITE               1.000000  0.800000  0.888889    5.000000
BLACK HAIRSTREAK            1.000000  1.000000  1.000000    5.000000
BLUE MORPHO                 1.000000  1.000000  1.000000    5.000000
BLUE SPOTTED CROW           0.833333  1.000000  0.909091    5.000000
BROWN SIPROETA              1.000000  1.000000  1.000000    5.000000